# Fine-Tuning GPT-2 on WebNLG with LoRA

Example training and evaluation commands for comparing attention-only LoRA with attention + MLP LoRA. See the [README](README.md) for project context, setup guidance, and archive limitations, and the [report](AML_project_report.pdf), Sections 3–4, for methodology and findings.


## Setup

The cells below download the upstream code, checkpoints, WebNLG data, and evaluation tools. Check installation output before proceeding: the saved run failed to install `torch==1.7.1+cu101`. The project's custom MLP changes are not included in the upstream clone.


In [ ]:
pwd

In [ ]:
!git clone https://github.com/microsoft/LoRA.git;

In [ ]:
!pip install loralib
!pip install torch transformers spacy tqdm tensorboard progress
!pip install nltk==3.5 pyter3==0.3 razdel==0.5.0 tabulate==0.8.7 bert-score==0.3.5

In [ ]:
%cd LoRA/examples/NLG
!pip install -r requirement.txt
!bash download_pretrained_checkpoints.sh
!bash create_datasets.sh
%cd ./eval
!bash download_evalscript.sh
%cd ..

## Training

Choose one configuration below; each trains for five epochs and writes checkpoints to `--work_dir`. Shared hyperparameters follow report Table 1.

The `wh_mlp` and `wo_mlp` filenames label historical variants; they do not change adapter placement. The exact MLP projections and ranks require the custom implementation, which is absent from this snapshot.


In [ ]:
pwd

'/content/LoRA/examples/NLG'

### GPT-2 Medium - rank 4

In [ ]:
import time
start = time.time()
!python -m torch.distributed.run --nproc_per_node=1 src/gpt2_ft.py \
    --train_data ./data/webnlg_challenge_2017/train.jsonl \
    --valid_data ./data/webnlg_challenge_2017/valid.jsonl \
    --train_batch_size 8 \
    --grad_acc 1 \
    --valid_batch_size 4 \
    --seq_len 512 \
    --model_card gpt2.md \
    --init_checkpoint ./pretrained_checkpoints/gpt2-medium-pytorch_model.bin \
    --platform local \
    --clip 0.0 \
    --lr 0.0002 \
    --weight_decay 0.01 \
    --correct_bias \
    --adam_beta2 0.999 \
    --scheduler linear \
    --warmup_step 500 \
    --max_epoch 5 \
    --save_interval 1000 \
    --lora_dim 4 \
    --lora_alpha 32 \
    --lora_dropout 0.1 \
    --label_smooth 0.1 \
    --work_dir /content/drive/MyDrive/1_trained_models/GPT2_M/webnlg \
    --random_seed 110
print(time.time() - start)

myrank: 0 local_rank: 0 device_count: 1 world_size: 1
        - platform : local
        - local_rank : 0
        - rank : 0
        - device : cuda:0
        - world_size : 1
        - random_seed : 110
        - lr : 0.0002
        - weight_decay : 0.01
        - correct_bias : True
        - adam_epislon : 1e-06
        - no_decay_bias : False
        - adam_beta1 : 0.9
        - adam_beta2 : 0.999
        - scheduler : linear
        - max_step : None
        - max_epoch : 5
        - warmup_step : 500
        - i_steps : 0
        - i_lrs : 0.00025
        - train_data : ./data/webnlg_challenge_2017/train.jsonl
        - valid_data : ./data/webnlg_challenge_2017/valid.jsonl
        - train_batch_size : 8
        - valid_batch_size : 4
        - grad_acc : 1
        - clip : 0.0
        - seq_len : 512
        - model_card : gpt2.md
        - init_checkpoint : ./pretrained_checkpoints/gpt2-medium-pytorch_model.bin
        - fp16 : False
        - log_interval : 100
        - eval_i

### GPT-2 Small — rank 8

In [ ]:
import time
start = time.time()
!python -m torch.distributed.run --nproc_per_node=1 src/gpt2_ft.py \
    --train_data ./data/webnlg_challenge_2017/train.jsonl \
    --valid_data ./data/webnlg_challenge_2017/valid.jsonl \
    --train_batch_size 8 \
    --grad_acc 1 \
    --valid_batch_size 4 \
    --seq_len 512 \
    --model_card gpt2.sm \
    --init_checkpoint ./pretrained_checkpoints/gpt2-pytorch_model.bin \
    --platform local \
    --clip 0.0 \
    --lr 0.0002 \
    --weight_decay 0.01 \
    --correct_bias \
    --adam_beta2 0.999 \
    --scheduler linear \
    --warmup_step 500 \
    --max_epoch 5 \
    --save_interval 1000 \
    --lora_dim 8 \
    --lora_alpha 32 \
    --lora_dropout 0.1 \
    --label_smooth 0.1 \
    --work_dir /content/drive/MyDrive/1_trained_models/GPT2_S/webnlg \
    --random_seed 110
print(time.time() - start)

myrank: 0 local_rank: 0 device_count: 1 world_size: 1
        - platform : local
        - local_rank : 0
        - rank : 0
        - device : cuda:0
        - world_size : 1
        - random_seed : 110
        - lr : 0.0002
        - weight_decay : 0.01
        - correct_bias : True
        - adam_epislon : 1e-06
        - no_decay_bias : False
        - adam_beta1 : 0.9
        - adam_beta2 : 0.999
        - scheduler : linear
        - max_step : None
        - max_epoch : 5
        - warmup_step : 500
        - i_steps : 0
        - i_lrs : 0.00025
        - train_data : ./data/webnlg_challenge_2017/train.jsonl
        - valid_data : ./data/webnlg_challenge_2017/valid.jsonl
        - train_batch_size : 8
        - valid_batch_size : 4
        - grad_acc : 1
        - clip : 0.0
        - seq_len : 512
        - model_card : gpt2.sm
        - init_checkpoint : ./pretrained_checkpoints/gpt2-pytorch_model.bin
        - fp16 : False
        - log_interval : 100
        - eval_interval

### GPT-2 Small — rank 16

In [ ]:
import time
start = time.time()
!python -m torch.distributed.run --nproc_per_node=1 src/gpt2_ft.py \
    --train_data ./data/webnlg_challenge_2017/train.jsonl \
    --valid_data ./data/webnlg_challenge_2017/valid.jsonl \
    --train_batch_size 8 \
    --grad_acc 1 \
    --valid_batch_size 4 \
    --seq_len 512 \
    --model_card gpt2.sm \
    --init_checkpoint ./pretrained_checkpoints/gpt2-pytorch_model.bin \
    --platform local \
    --clip 0.0 \
    --lr 0.0002 \
    --weight_decay 0.01 \
    --correct_bias \
    --adam_beta2 0.999 \
    --scheduler linear \
    --warmup_step 500 \
    --max_epoch 5 \
    --save_interval 1000 \
    --lora_dim 16 \
    --lora_alpha 32 \
    --lora_dropout 0.1 \
    --label_smooth 0.1 \
    --work_dir /content/drive/MyDrive/1_trained_models/GPT2_S/webnlg-r16 \
    --random_seed 110
print(time.time() - start)

myrank: 0 local_rank: 0 device_count: 1 world_size: 1
        - platform : local
        - local_rank : 0
        - rank : 0
        - device : cuda:0
        - world_size : 1
        - random_seed : 110
        - lr : 0.0002
        - weight_decay : 0.01
        - correct_bias : True
        - adam_epislon : 1e-06
        - no_decay_bias : False
        - adam_beta1 : 0.9
        - adam_beta2 : 0.999
        - scheduler : linear
        - max_step : None
        - max_epoch : 5
        - warmup_step : 500
        - i_steps : 0
        - i_lrs : 0.00025
        - train_data : ./data/webnlg_challenge_2017/train.jsonl
        - valid_data : ./data/webnlg_challenge_2017/valid.jsonl
        - train_batch_size : 8
        - valid_batch_size : 4
        - grad_acc : 1
        - clip : 0.0
        - seq_len : 512
        - model_card : gpt2.sm
        - init_checkpoint : ./pretrained_checkpoints/gpt2-pytorch_model.bin
        - fp16 : False
        - log_interval : 100
        - eval_interval

## Generating predictions

Select the command matching your model, rank, and adapter placement. The examples below are Small/rank 8, Medium/rank 4, two Small/rank 4 variants, and Small/rank 16.

Replace `--init_checkpoint` with the checkpoint from your run. The historical renamed checkpoints and Small/rank 4 training steps are not shown here. Predictions are written to `--work_dir/--output_file`.


In [ ]:
pwd

'/content/LoRA/examples/NLG'

In [ ]:
# %cd

In [ ]:
import time
start = time.time()
!python -m torch.distributed.run --nproc_per_node=1 src/gpt2_beam.py \
    --data ./data/webnlg_challenge_2017/test.jsonl \
    --batch_size 1 \
    --seq_len 512 \
    --eval_len 64 \
    --model_card gpt2.sm \
    --init_checkpoint /content/drive/MyDrive/1_trained_models/GPT2_S/webnlg/model.11270.pt \
    --platform local \
    --lora_dim 8 \
    --lora_alpha 32 \
    --beam 10 \
    --length_penalty 0.8 \
    --no_repeat_ngram_size 4 \
    --repetition_penalty 1.0 \
    --eos_token_id 628 \
    --work_dir /content/drive/MyDrive/1_trained_models/GPT2_S/webnlg \
    --output_file predict.gpt2_fu_sm_lora_webnlg_wh_mlp.b1dim8.jsonl
print(time.time() - start)

myrank: 0 local_rank: 0 device_count: 1 world_size: 1
        - platform : local
        - local_rank : 0
        - rank : 0
        - device : cuda:0
        - world_size : 1
        - random_seed : 10
        - data : ./data/webnlg_challenge_2017/test.jsonl
        - batch_size : 1
        - seq_len : 512
        - eval_len : 64
        - min_length : 0
        - model_card : gpt2.sm
        - init_checkpoint : /content/drive/MyDrive/1_trained_models/GPT2_S/webnlg/model.11270.pt
        - lora_dim : 8
        - lora_alpha : 32
        - work_dir : /content/drive/MyDrive/1_trained_models/GPT2_S/webnlg
        - beam : 10
        - length_penalty : 0.8
        - no_repeat_ngram_size : 4
        - repetition_penalty : 1.0
        - eos_token_id : [50256, 628]
        - output_file : predict.gpt2_fu_sm_lora_webnlg_wh_mlp.b1dim8.jsonl
        - dist : <module 'torch.distributed' from '/usr/local/lib/python3.10/dist-packages/torch/distributed/__init__.py'>
Experiment dir : /content/drive/M

In [ ]:
import time
start = time.time()
!python -m torch.distributed.run --nproc_per_node=1 src/gpt2_beam.py \
    --data ./data/webnlg_challenge_2017/test.jsonl \
    --batch_size 1 \
    --seq_len 512 \
    --eval_len 64 \
    --model_card gpt2.md \
    --init_checkpoint /content/drive/MyDrive/1_trained_models/GPT2_M/webnlg/model.11270_md_MLP.pt \
    --platform local \
    --lora_dim 4 \
    --lora_alpha 32 \
    --beam 10 \
    --length_penalty 0.8 \
    --no_repeat_ngram_size 4 \
    --repetition_penalty 1.0 \
    --eos_token_id 628 \
    --work_dir /content/drive/MyDrive/1_trained_models/GPT2_M/webnlg \
    --output_file predict.gpt2_fu_md_lora_webnlg_wh_mlp_NEW_b1.b10p08r4.jsonl
print(time.time() - start)

myrank: 0 local_rank: 0 device_count: 1 world_size: 1
        - platform : local
        - local_rank : 0
        - rank : 0
        - device : cuda:0
        - world_size : 1
        - random_seed : 10
        - data : ./data/webnlg_challenge_2017/test.jsonl
        - batch_size : 1
        - seq_len : 512
        - eval_len : 64
        - min_length : 0
        - model_card : gpt2.md
        - init_checkpoint : /content/drive/MyDrive/1_trained_models/GPT2_M/webnlg/model.11270_md_MLP.pt
        - lora_dim : 4
        - lora_alpha : 32
        - work_dir : /content/drive/MyDrive/1_trained_models/GPT2_M/webnlg
        - beam : 10
        - length_penalty : 0.8
        - no_repeat_ngram_size : 4
        - repetition_penalty : 1.0
        - eos_token_id : [50256, 628]
        - output_file : predict.gpt2_fu_md_lora_webnlg_wh_mlp_NEW_b1.b10p08r4.jsonl
        - dist : <module 'torch.distributed' from '/usr/local/lib/python3.10/dist-packages/torch/distributed/__init__.py'>
Experiment dir : 

In [ ]:
import time
start = time.time()
!python -m torch.distributed.run --nproc_per_node=1 src/gpt2_beam.py \
    --data ./data/webnlg_challenge_2017/test.jsonl \
    --batch_size 4 \
    --seq_len 512 \
    --eval_len 64 \
    --model_card gpt2.sm \
    --init_checkpoint /content/drive/MyDrive/1_trained_models/GPT2_M/webnlg/model.11270_sm.pt \
    --platform local \
    --lora_dim 4 \
    --lora_alpha 32 \
    --beam 10 \
    --length_penalty 0.8 \
    --no_repeat_ngram_size 4 \
    --repetition_penalty 1.0 \
    --eos_token_id 628 \
    --work_dir /content/drive/MyDrive/1_trained_models/GPT2_M/webnlg \
    --output_file predict.gpt2_fu_sm_lora_webnlg_wo_mlp.b10p08r4.jsonl
print(time.time() - start)

myrank: 0 local_rank: 0 device_count: 1 world_size: 1
        - platform : local
        - local_rank : 0
        - rank : 0
        - device : cuda:0
        - world_size : 1
        - random_seed : 10
        - data : ./data/webnlg_challenge_2017/test.jsonl
        - batch_size : 4
        - seq_len : 512
        - eval_len : 64
        - min_length : 0
        - model_card : gpt2.sm
        - init_checkpoint : /content/drive/MyDrive/1_trained_models/GPT2_M/webnlg/model.11270_sm.pt
        - lora_dim : 4
        - lora_alpha : 32
        - work_dir : /content/drive/MyDrive/1_trained_models/GPT2_M/webnlg
        - beam : 10
        - length_penalty : 0.8
        - no_repeat_ngram_size : 4
        - repetition_penalty : 1.0
        - eos_token_id : [50256, 628]
        - output_file : predict.gpt2_fu_sm_lora_webnlg_wo_mlp.b10p08r4.jsonl
        - dist : <module 'torch.distributed' from '/usr/local/lib/python3.10/dist-packages/torch/distributed/__init__.py'>
Experiment dir : /content/dr

In [ ]:
import time
start = time.time()
!python -m torch.distributed.run --nproc_per_node=1 src/gpt2_beam.py \
    --data ./data/webnlg_challenge_2017/test.jsonl \
    --batch_size 4 \
    --seq_len 512 \
    --eval_len 64 \
    --model_card gpt2.sm \
    --init_checkpoint /content/drive/MyDrive/1_trained_models/GPT2_M/webnlg/model.11270_sm_MLP.pt \
    --platform local \
    --lora_dim 4 \
    --lora_alpha 32 \
    --beam 10 \
    --length_penalty 0.8 \
    --no_repeat_ngram_size 4 \
    --repetition_penalty 1.0 \
    --eos_token_id 628 \
    --work_dir /content/drive/MyDrive/1_trained_models/GPT2_M/webnlg \
    --output_file predict.gpt2_fu_sm_lora_webnlg_wh_mlp.b10p08r4.jsonl
print(time.time() - start)

myrank: 0 local_rank: 0 device_count: 1 world_size: 1
        - platform : local
        - local_rank : 0
        - rank : 0
        - device : cuda:0
        - world_size : 1
        - random_seed : 10
        - data : ./data/webnlg_challenge_2017/test.jsonl
        - batch_size : 4
        - seq_len : 512
        - eval_len : 64
        - min_length : 0
        - model_card : gpt2.sm
        - init_checkpoint : /content/drive/MyDrive/1_trained_models/GPT2_M/webnlg/model.11270_sm_MLP.pt
        - lora_dim : 4
        - lora_alpha : 32
        - work_dir : /content/drive/MyDrive/1_trained_models/GPT2_M/webnlg
        - beam : 10
        - length_penalty : 0.8
        - no_repeat_ngram_size : 4
        - repetition_penalty : 1.0
        - eos_token_id : [50256, 628]
        - output_file : predict.gpt2_fu_sm_lora_webnlg_wh_mlp.b10p08r4.jsonl
        - dist : <module 'torch.distributed' from '/usr/local/lib/python3.10/dist-packages/torch/distributed/__init__.py'>
Experiment dir : /conten

In [ ]:
import time
start = time.time()
!python -m torch.distributed.run --nproc_per_node=1 src/gpt2_beam.py \
    --data ./data/webnlg_challenge_2017/test.jsonl \
    --batch_size 1 \
    --seq_len 512 \
    --eval_len 64 \
    --model_card gpt2.sm \
    --init_checkpoint /content/drive/MyDrive/1_trained_models/GPT2_S/webnlg-r16/model.11270.pt \
    --platform local \
    --lora_dim 16 \
    --lora_alpha 32 \
    --beam 10 \
    --length_penalty 0.8 \
    --no_repeat_ngram_size 4 \
    --repetition_penalty 1.0 \
    --eos_token_id 628 \
    --work_dir /content/drive/MyDrive/1_trained_models/GPT2_S/webnlg-r16 \
    --output_file predict.gpt2_fu_sm_lora_webnlg_wh_mlp.b1dim16.jsonl
print(time.time() - start)

myrank: 0 local_rank: 0 device_count: 1 world_size: 1
        - platform : local
        - local_rank : 0
        - rank : 0
        - device : cuda:0
        - world_size : 1
        - random_seed : 10
        - data : ./data/webnlg_challenge_2017/test.jsonl
        - batch_size : 1
        - seq_len : 512
        - eval_len : 64
        - min_length : 0
        - model_card : gpt2.sm
        - init_checkpoint : /content/drive/MyDrive/1_trained_models/GPT2_S/webnlg-r16/model.11270.pt
        - lora_dim : 16
        - lora_alpha : 32
        - work_dir : /content/drive/MyDrive/1_trained_models/GPT2_S/webnlg-r16
        - beam : 10
        - length_penalty : 0.8
        - no_repeat_ngram_size : 4
        - repetition_penalty : 1.0
        - eos_token_id : [50256, 628]
        - output_file : predict.gpt2_fu_sm_lora_webnlg_wh_mlp.b1dim16.jsonl
        - dist : <module 'torch.distributed' from '/usr/local/lib/python3.10/dist-packages/torch/distributed/__init__.py'>
Experiment dir : /conte

## Decoding predictions and computing metrics

The example decodes the Medium/MLP predictions and evaluates them against WebNLG references. Update the prediction, hypothesis, and reference paths together for another configuration.

The saved evaluation contains a test/reference length mismatch, METEOR -1, and BLEU warnings; these are not valid results for comparison with the report.


In [ ]:
pwd

'/content/LoRA/examples/NLG'

In [ ]:
!python src/gpt2_decode.py \
    --vocab ./vocab \
    --sample_file /content/drive/MyDrive/1_trained_models/GPT2_M/webnlg/predict.gpt2_fu_md_lora_webnlg_wh_mlp_NEW_b1.b10p08r4.jsonl \
    --input_file ./data/webnlg_challenge_2017/test_formatted.jsonl \
    --ref_type webnlg \
    --ref_num 6 \
    --output_ref_file /content/drive/MyDrive/1_eval/md_wh_mlp/references_webnlg \
    --output_pred_file /content/drive/MyDrive/1_eval/md_wh_mlp/hypothesis_webnlg \
    --tokenize --lower

unique refer dict 1862


In [ ]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
%cd ./eval/GenerationEval/
!python eval.py \
    -R /content/drive/MyDrive/1_eval/md_wh_mlp/references_webnlg/reference \
    -H /content/drive/MyDrive/1_eval/md_wh_mlp/hypothesis_webnlg \
    -nr 6 \
    -m bleu,meteor,ter
%cd ../..

/content/LoRA/examples/NLG/eval/GenerationEval
2024-04-22 22:00:49.522224: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-04-22 22:00:49.522281: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-04-22 22:00:49.524178: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-04-22 22:00:50.580137: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
STARTING TO PARSE INPUTS...
FINISHING TO PARSE INPUTS...
STARTING TO COMPUTE BLEU...
Use of uninitialized value in division (/) at metrics/multi-bleu-detok.perl line 146, <STDIN> line 1862.
Use 

## Results

See [report Section 4](AML_project_report.pdf) for parameter counts (Table 2), loss and perplexity (Figures 1–3), and generation metrics (Figure 4). The reported attention + MLP variant performed worse under the tested settings; this observation does not establish a general effect of MLP LoRA.
